# Lab 6 part 1
## Task 1

In [2]:
import pandas as pd

listings = pd.read_csv("../data/listings_clean.csv")

In [3]:
reviews = pd.Series([90, 97, 100, 102, 107])
print("Mean", reviews.mean())
print("Median", reviews.median())

Mean 99.2
Median 100.0


In [4]:
reviews = pd.Series([90, 97, 100, 102, 107, 1293])
print("Mean", reviews.mean())
print("Median", reviews.median())

Mean 298.1666666666667
Median 101.0


When adding the last large datapoint the mean increased significantly, while the median remained about the same.

In [6]:
print("Mean for listings", listings["number_of_reviews"].mean())
print("Median for listings", listings["number_of_reviews"].median())
print(listings["number_of_reviews"].describe())

Mean for listings 35.75404459991255
Median for listings 9.0
count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64


Given that the max value is significantly larger than the majority of the datapoints, I would choose the median as the value to best answer the question "How many reviews does a typical listing have?".

## Task 2

In [7]:
listings["number_of_reviews"].describe()

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64

In [9]:
listings['number_of_reviews'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(0)

0.10      0.0
0.25      2.0
0.50      9.0
0.75     29.0
0.90     94.0
0.99    400.0
Name: number_of_reviews, dtype: float64

In [13]:
q1 = listings['number_of_reviews'].quantile(0.25)
q3 = listings['number_of_reviews'].quantile(0.75)

iqr = q3 - q1

print("q1", q1)
print("q3", q3)
print("Interquartile range", iqr)

q1 2.0
q3 29.0
Interquartile range 27.0


The 10th percentile being 0 tells us that (at least) 10% of the listings have 0 reviews.

The IQR tells me about the middle 50% of a dataset, and how spread out it is.

## Task 3

In [14]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

print(upper, lower)

69.5 -38.5


For this dataset there cannot be any outliers below the lower fence, since the lower fence is negative and the dataset values cannot take on negative values.

In [17]:
upper_outliers = listings[listings["number_of_reviews"] > upper]
upper_outliers["number_of_reviews"].describe()

count     582.000000
mean      192.187285
std       136.400813
min        70.000000
25%        98.000000
50%       148.500000
75%       236.500000
max      1293.000000
Name: number_of_reviews, dtype: float64

In [23]:
upper_outliers["room_type"].value_counts()

room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64

In [27]:
non_outliers = listings[listings["number_of_reviews"] <= upper]["price"]
print("Median price, upper outliers", upper_outliers["price"].median())
print("Median price, all other listings", non_outliers.median())

Median price, upper outliers 1459.54
Median price, all other listings 1867.0


## Task 4

In [ ]:
listings["reviews_outlier"] = listings["number_of_reviews"] > upper
listings["reviews_outlier"].value_counts()

reviews_outlier
False    3992
True      582
Name: count, dtype: int64

In [49]:
data = {
    "all_listings": [
        listings["number_of_reviews"].count(),
        listings["number_of_reviews"].mean(),
        listings["number_of_reviews"].median(),
        listings["number_of_reviews"].std(),
        ],
    "without_outliers": [
        listings[~listings["reviews_outlier"]]["number_of_reviews"].count(),
        listings[~listings["reviews_outlier"]]["number_of_reviews"].mean(),
        listings[~listings["reviews_outlier"]]["number_of_reviews"].median(),
        listings[~listings["reviews_outlier"]]["number_of_reviews"].std(),
        ],
}

table = pd.DataFrame(data, index = ["count", "mean", "median", "std"])

table.round(1)


,all_listings,without_outliers
count,4574.0,3992.0
mean,35.8,12.9
median,9.0,7.0
std,78.4,15.5


In [61]:
count_percentage = (table["without_outliers"]["count"] / table["all_listings"]["count"] - 1) * 100
mean_percentage = (table["without_outliers"]["mean"] / table["all_listings"]["mean"] - 1) * 100
median_percentage = (table["without_outliers"]["median"] / table["all_listings"]["median"] - 1) * 100
std_percentage = (table["without_outliers"]["std"] / table["all_listings"]["std"] - 1) * 100


print("Count percentage change", count_percentage.round(2))
print("Mean percentage change", mean_percentage.round(2))
print("Median percentage change", median_percentage.round(2))
print("Standard deviation percentage change", std_percentage.round(2))

Count percentage change -12.72
Mean percentage change -63.79
Median percentage change -22.22
Standard deviation percentage change -80.18


The standard deviation changes the most and the count changes the least.

## Task 5

In [63]:
listings.groupby("room_type")["number_of_reviews"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
room_type,,,,,,,,
Entire home/apt,3760.0,34.0,76.0,0.0,2.0,8.0,27.0,1293.0
Hotel room,3.0,61.0,16.0,49.0,52.0,56.0,68.0,79.0
Private room,787.0,41.0,84.0,0.0,2.0,12.0,38.0,867.0
Shared room,24.0,114.0,122.0,0.0,28.0,82.0,146.0,510.0


The room type with the highest median (50% value) is shared room.  
The hotel room type has only 3 datapoints, so I would not say anything about that category.

## Task 6

In [85]:
columns = ["price", "accommodates", "bedrooms", "availability_365", 
           "review_scores_rating", "minimum_nights", "days_since_review"]

for column in columns:
    print("Correlation for number of reviews and", column,  
          listings["number_of_reviews"].corr(listings[column]).round(2))

Correlation for number of reviews and price -0.09
Correlation for number of reviews and accommodates -0.05
Correlation for number of reviews and bedrooms -0.14
Correlation for number of reviews and availability_365 0.18
Correlation for number of reviews and review_scores_rating -0.04
Correlation for number of reviews and minimum_nights -0.07
Correlation for number of reviews and days_since_review -0.19


Strongest correlation: days_since_review  
Weakest correlation: review_scores_rating  
Negative correlation: price, accommodates, bedrooms, review_scores_rating, minimum_nights, days_since_review


In [87]:
print("With outliers: Correlation for number of reviews and availability 365:", 
    listings["number_of_reviews"].corr(listings["availability_365"]).round(2))

print("With outliers: Correlation for number of reviews and availability 365:", 
    listings[~listings["reviews_outlier"]]["number_of_reviews"].corr(listings["availability_365"]).round(2))

With outliers: Correlation for number of reviews and availability 365: 0.18
With outliers: Correlation for number of reviews and availability 365: 0.0


The correlation goes from 0.18 to 0.0 when removing the outliers.  
The correlation was weak before, and after the outlier removal there is no correlation.